# Bioserenity — all metrics (metadata + YASA sleep stats + infraslow)

For every subject that has **all three** inputs on `$OAK`

* metadata row in `Excel/bioserenity_metadata3.csv`,
* an EDF at `edf/{id}.edf`,
* a hypnodensity at `Sleep_Staging/{id}_Hypnodensity.csv`,

this notebook computes one row of metrics and exports a single merged CSV.

The heavy lifting lives in reusable functions in
[`infraslow.processing.all_metrics`](infraslow/processing/all_metrics.py); this
notebook just wires paths, calls them, and reports. It reproduces the
single-subject analysis of `demo_infraslow_yasa_average.ipynb` at cohort scale.

**Output columns** (52): `ID, Age, Gender, BMI` · the full YASA `sleep_statistics`
(`TIB, SPT, WASO, TST, N1, N2, N3, REM, NREM, SOL, Lat_*, %*, SE, SME`) · and for
each of `N2 / N3 / NREM`: `peak_freq(_sem)`, `fit_peak_freq(_sem)`, `bandwidth_hz`,
`auc`, `detected`, `bouts` (min), `%bouts` (of TST).

**Conventions** (see the module docstring for the full rationale):
* `peak_freq` = empirical argmax of the baseline-corrected relative spectrum;
  `fit_peak_freq` = Gaussian-fit centre. Both averaged across bouts; `_sem` is the
  standard error across bouts (`NaN` if <2 bouts).
* `bandwidth_hz`, `auc`, `detected` come from the fit of the bout-averaged
  spectrum (as in the reference notebook); `detected` = peak > 1.5·SD(baseline).
* `NREM` = N1+N2+N3 (matches the YASA `NREM` column); spindles for the bout filter
  are detected in N2+N3.
* `{stage}_bouts` = total minutes of consecutive stage runs ≥ 200 s; `%{stage}_bouts`
  = that as a percentage of **TST**.

> ⚠️ **Run this on a compute node** (`sh_dev` / `salloc` / `sbatch`), **not the
> login node**: it imports `lunapi` (EDF I/O) and `yasa`, and reads large EDFs.

## 1. Imports and configuration

In [1]:
import logging
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Make the src-layout `infraslow` package importable regardless of the kernel CWD.
def _find_up(marker: str) -> Path:
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if (cand / marker).exists():
            return cand
    raise FileNotFoundError(f'Could not find {marker!r} above {here}')

REPO_ROOT = _find_up('pyproject.toml')
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from infraslow.io.hypnodensity import DEFAULT_HYPNODENSITY_SUFFIX
from infraslow.processing.all_metrics import (
    all_metric_columns,
    calculate_subject_all_metrics,
    find_valid_bioserenity_subjects,
    load_bioserenity_metadata,
    run_all_subjects,
    CHANNEL,
    SF,
    INFRASLOW_STAGES,
)

# INFO-level logs surface the per-subject OK/FAIL lines from run_all_subjects.
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s',
    force=True,
)
logging.getLogger('infraslow').setLevel(logging.INFO)
pd.set_option('display.max_columns', 100)

print(f'repo root : {REPO_ROOT}')
print(f'channel   : {CHANNEL} @ {SF:g} Hz   infraslow stages: {INFRASLOW_STAGES}')

repo root : /home/users/chaisaen/infraslow
channel   : C3 @ 128 Hz   infraslow stages: ('N2', 'N3', 'NREM')


## 2. Path setup using `$OAK`

In [2]:
oak_raw = os.environ.get('OAK')
if not oak_raw:
    raise EnvironmentError('$OAK is not set. Load your environment before running.')
OAK = Path(oak_raw)

BIOSERENITY = OAK / 'psg' / 'Bioserenity'
METADATA_PATH = BIOSERENITY / 'Excel' / 'bioserenity_metadata3.csv'
EDF_DIR = BIOSERENITY / 'edf'
STAGING_DIR = BIOSERENITY / 'Sleep_Staging'

# The repo ships an `output/` folder; reuse it (the prompt's `outputs/` -> `output/`).
OUTPUT_DIR = REPO_ROOT / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)
FINAL_CSV = OUTPUT_DIR / 'bioserenity_all_metrics.csv'
FAILED_CSV = OUTPUT_DIR / 'bioserenity_all_metrics_failed_subjects.csv'

for label, path in [
    ('metadata', METADATA_PATH), ('edf dir', EDF_DIR), ('staging dir', STAGING_DIR),
]:
    status = 'OK' if path.exists() else 'MISSING'
    print(f'{label:<12}: [{status}] {path}')
print(f'{"output":<12}: {OUTPUT_DIR}')

metadata    : [OK] /oak/stanford/groups/mignot/psg/Bioserenity/Excel/bioserenity_metadata3.csv
edf dir     : [OK] /oak/stanford/groups/mignot/psg/Bioserenity/edf
staging dir : [OK] /oak/stanford/groups/mignot/psg/Bioserenity/Sleep_Staging
output      : /home/users/chaisaen/infraslow/output


## 3. Load metadata

Reads only `ID, Age, Gender, BMI` (the CSV is wide and long), tolerating minor
header-name differences.

In [3]:
metadata = load_bioserenity_metadata(METADATA_PATH)
print(f'metadata rows: {len(metadata):,}')
metadata.head()

metadata rows: 73,223


,ID,Age,Gender,BMI
0,{BCDDD1AE-F41D-41A9-9626-7DCF98107279},49,M,30.10
1,{9E64BABE-51FD-4E92-8E3B-61D73382C35F},64,M,34.08
2,{4ADD7F52-EA02-46E4-B214-0EDF3EAB1D7D},60,M,32.43
3,{3A8B7904-473F-4DE3-AEB4-93161DCF4E4A},55,M,29.98
4,{FC65CA3A-B877-45EF-9F5B-9CBCBAEB7B8C},67,F,37.64


## 4. Find valid subjects (matching EDF + hypnodensity)

Lists each directory once (single `readdir`) and tests membership in memory —
no per-subject `stat` storm against Lustre.

In [4]:
valid = find_valid_bioserenity_subjects(metadata, EDF_DIR, STAGING_DIR)
availability = valid.attrs['availability']
print('availability:')
for key, value in availability.items():
    print(f'  {key:<22}: {value:,}')
valid.head()

availability:
  n_metadata            : 73,223
  n_with_edf            : 73,223
  n_with_hypnodensity   : 59,933
  n_valid               : 59,933


,ID,Age,Gender,BMI
0,{BCDDD1AE-F41D-41A9-9626-7DCF98107279},49,M,30.10
1,{9E64BABE-51FD-4E92-8E3B-61D73382C35F},64,M,34.08
2,{4ADD7F52-EA02-46E4-B214-0EDF3EAB1D7D},60,M,32.43
3,{3A8B7904-473F-4DE3-AEB4-93161DCF4E4A},55,M,29.98
4,{FC65CA3A-B877-45EF-9F5B-9CBCBAEB7B8C},67,F,37.64


## 5. Helper functions (imported from `infraslow.processing.all_metrics`)

The pipeline is built from these reusable functions (all documented in the module):

| Function | Role |
|---|---|
| `load_bioserenity_metadata` | metadata → standardized `ID, Age, Gender, BMI` |
| `find_valid_bioserenity_subjects` | keep subjects with both EDF + hypnodensity |
| `load_hypnodensity_as_hypnogram` | hypnodensity CSV → YASA integer hypnogram (argmax) |
| `calculate_yasa_sleep_statistics` | full YASA `sleep_statistics` |
| `calculate_stage_bouts` | consolidated-bout duration per stage |
| `calculate_stage_infraslow` | infraslow peak/bandwidth/AUC/detection per stage |
| `calculate_subject_infraslow_metrics` | all stage-groups for one subject |
| `calculate_subject_all_metrics` | metadata + YASA + infraslow for one subject |
| `run_all_subjects` | cohort runner with per-subject error isolation |

## 6. Test the full pipeline on one subject

Sanity-check the end-to-end computation on the first valid subject before the
full run. (This reads one EDF — expect it to take a little while.)

In [5]:
if valid.empty:
    raise RuntimeError('No valid subjects found — check the $OAK paths above.')

test_row = valid.iloc[0]
test_id = test_row['ID']
test_edf = EDF_DIR / f'{test_id}.edf'
test_hypno = STAGING_DIR / f'{test_id}{DEFAULT_HYPNODENSITY_SUFFIX}'
print(f'testing subject: {test_id}')

one_row = calculate_subject_all_metrics(test_row, test_edf, test_hypno)
pd.Series(one_row)[all_metric_columns()].to_frame(name=test_id)

testing subject: {BCDDD1AE-F41D-41A9-9626-7DCF98107279}


2026-07-08 14:01:20,098 INFO infraslow.io.psg_loader: Loading subject {BCDDD1AE-F41D-41A9-9626-7DCF98107279} from /oak/stanford/groups/mignot/psg/Bioserenity/edf/{BCDDD1AE-F41D-41A9-9626-7DCF98107279}.edf
___________________________________________________________________
2026-07-08 14:01:28,640 INFO infraslow.io.psg_loader: EDF exposes 26 physical channels.


initiated lunapi 1.4.0 <lunapi.lunapi0.luna object at 0x7fdcd63adcf0> 



Processing: {BCDDD1AE-F41D-41A9-9626-7DCF98107279} | /oak/stanford/groups/mignot/psg/Bioserenity/edf/{BCDDD1AE-F41D-41A9-9626-7DCF98107279}.edf
 duration 07.06.33, 25593s | time 21.58.25 - 05.04.58 | date 01.01.12

 signals: 27 (of 27) selected in an EDF+C file
  F3 | F4 | C3 | C4 | O1 | O2 | LOC | ROC
  ECG | emg_Chin | emg_RLeg | emg_LLeg | Tho | Abd | SpO2 | flow_PFlo
  flow_TFlo | flow_CFlo | Snore | IPAP | EPAP | Leak | Body | Pulse
  sao2 | HR | EDF Annotations
  extracting 'EDF Annotations' track from EDF+
2026-07-08 14:01:30,064 INFO infraslow.io.psg_loader: Loaded 1 channel(s), 3275904 sample(s); 0 missing, 0 conflict record(s).
2026-07-08 14:01:35,148 WARNING yasa: Hypnogram is SHORTER than data by 3.00 seconds. Padding hypnogram with Unscored (UNS) to match data.size.


Setting up band-pass filter from 1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 423 samples (3.305 s)

Setting up band-pass filter from 12 - 15 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 12.00
- Lower transition bandwidth: 1.50 Hz (-6 dB cutoff frequency: 11.25 Hz)
- Upper passband edge: 15.00 Hz
- Upper transition bandwidth: 1.50 Hz (-6 dB cutoff frequency: 15.75 Hz)
- F

,{BCDDD1AE-F41D-41A9-9626-7DCF98107279}
ID,{BCDDD1AE-F41D-41A9-9626-7DCF98107279}
Age,49
Gender,M
BMI,30.1
TIB,426.5
SPT,402.5
WASO,29.5
TST,373.0
N1,23.5
N2,201.5


## 7. Run the pipeline for all valid subjects

Set `MAX_SUBJECTS` to a small integer for a quick smoke test; leave it `None` to
process the **entire** valid cohort. Each subject is isolated in a `try/except`,
so a single bad recording is recorded and skipped rather than aborting the run.
For the full cohort, submit this notebook as a batch job (`papermill` / `jupyter
nbconvert --execute`) with enough walltime and memory.

In [6]:
MAX_SUBJECTS = None  # e.g. 10 for a quick test; None = all valid subjects

subset = valid if MAX_SUBJECTS is None else valid.head(MAX_SUBJECTS)
print(f'processing {len(subset):,} subject(s)...')

results_df, failed_df = run_all_subjects(subset, EDF_DIR, STAGING_DIR)
print(f'\nprocessed OK : {len(results_df):,}')
print(f'failed       : {len(failed_df):,}')

processing 59,933 subject(s)...


Subjects:   0%|          | 0/59933 [00:00<?, ?subj/s]2026-07-08 14:02:38,674 INFO infraslow.io.psg_loader: Loading subject {BCDDD1AE-F41D-41A9-9626-7DCF98107279} from /oak/stanford/groups/mignot/psg/Bioserenity/edf/{BCDDD1AE-F41D-41A9-9626-7DCF98107279}.edf
___________________________________________________________________
Processing: {BCDDD1AE-F41D-41A9-9626-7DCF98107279} | /oak/stanford/groups/mignot/psg/Bioserenity/edf/{BCDDD1AE-F41D-41A9-9626-7DCF98107279}.edf
 duration 07.06.33, 25593s | time 21.58.25 - 05.04.58 | date 01.01.12

 signals: 27 (of 27) selected in an EDF+C file
  F3 | F4 | C3 | C4 | O1 | O2 | LOC | ROC
  ECG | emg_Chin | emg_RLeg | emg_LLeg | Tho | Abd | SpO2 | flow_PFlo
  flow_TFlo | flow_CFlo | Snore | IPAP | EPAP | Leak | Body | Pulse
  sao2 | HR | EDF Annotations
  extracting 'EDF Annotations' track from EDF+
2026-07-08 14:02:38,973 INFO infraslow.io.psg_loader: EDF exposes 26 physical channels.


initiated lunapi 1.4.0 <lunapi.lunapi0.luna object at 0x7fdcd63adcf0> 



2026-07-08 14:02:39,965 INFO infraslow.io.psg_loader: Loaded 1 channel(s), 3275904 sample(s); 0 missing, 0 conflict record(s).
2026-07-08 14:02:42,370 WARNING yasa: Hypnogram is SHORTER than data by 3.00 seconds. Padding hypnogram with Unscored (UNS) to match data.size.


Setting up band-pass filter from 1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 423 samples (3.305 s)

Setting up band-pass filter from 12 - 15 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 12.00
- Lower transition bandwidth: 1.50 Hz (-6 dB cutoff frequency: 11.25 Hz)
- Upper passband edge: 15.00 Hz
- Upper transition bandwidth: 1.50 Hz (-6 dB cutoff frequency: 15.75 Hz)
- F

2026-07-08 14:02:46,947 INFO infraslow.processing.all_metrics: OK   {BCDDD1AE-F41D-41A9-9626-7DCF98107279}
Subjects:   0%|          | 1/59933 [00:08<137:54:44,  8.28s/subj]2026-07-08 14:02:47,023 INFO infraslow.io.psg_loader: Loading subject {9E64BABE-51FD-4E92-8E3B-61D73382C35F} from /oak/stanford/groups/mignot/psg/Bioserenity/edf/{9E64BABE-51FD-4E92-8E3B-61D73382C35F}.edf
___________________________________________________________________
2026-07-08 14:02:48,617 INFO infraslow.io.psg_loader: EDF exposes 28 physical channels.
Processing: {9E64BABE-51FD-4E92-8E3B-61D73382C35F} | /oak/stanford/groups/mignot/psg/Bioserenity/edf/{9E64BABE-51FD-4E92-8E3B-61D73382C35F}.edf
 duration 08.04.26, 29066s | time 21.45.25 - 05.49.51 | date 01.01.13

 signals: 29 (of 29) selected in an EDF+C file
  F3 | F4 | C3 | C4 | O1 | O2 | LOC | ROC
  ECG | emg_Chin | emg_RLeg | emg_LLeg | Tho | Abd | SpO2 | flow_PFlo
  flow_TFlo | flow_CFLO | PPG | Snore | imp | rr | Leak | IPAP
  EPAP | Body | sao2 | HR | ED

initiated lunapi 1.4.0 <lunapi.lunapi0.luna object at 0x7fdcd63adcf0> 



2026-07-08 14:02:53,487 INFO infraslow.io.psg_loader: Loaded 1 channel(s), 3720448 sample(s); 0 missing, 0 conflict record(s).
Subjects:   0%|          | 1/59933 [00:17<293:17:49, 17.62s/subj]


KeyboardInterrupt: 

## 8. Save the final CSV(s)

In [ ]:
results_df.to_csv(FINAL_CSV, index=False)
failed_df.to_csv(FAILED_CSV, index=False)
print(f'metrics       -> {FINAL_CSV}')
print(f'failed subjects -> {FAILED_CSV}')

## 9. Processing summary

In [ ]:
print('=' * 60)
print('PROCESSING SUMMARY')
print('=' * 60)
print(f'Metadata subjects          : {availability["n_metadata"]:,}')
print(f'  with EDF file            : {availability["n_with_edf"]:,}')
print(f'  with hypnodensity file   : {availability["n_with_hypnodensity"]:,}')
print(f'Valid (both files present) : {availability["n_valid"]:,}')
print(f'Attempted this run         : {len(subset):,}')
print(f'Successfully processed     : {len(results_df):,}')
print(f'Failed                     : {len(failed_df):,}')
print('-' * 60)
if not failed_df.empty:
    print('Failures by error_type:')
    for etype, count in failed_df['error_type'].value_counts().items():
        print(f'  {etype:<28}: {count}')
    print('-' * 60)
print(f'Final metrics CSV : {FINAL_CSV}')
print(f'Failed subjects   : {FAILED_CSV}')
print('=' * 60)

## 10. Sanity checks: `head()` and `describe()`

In [ ]:
results_df.head()

In [ ]:
results_df.describe(include='all').T

In [ ]:
# Detection rate per stage (quick quality read-out).
for stage in INFRASLOW_STAGES:
    col = f'{stage}_detected'
    if col in results_df.columns and len(results_df):
        rate = results_df[col].mean()
        print(f'{stage:<5} infraslow detected: {rate:6.1%}  ({int(results_df[col].sum())}/{len(results_df)})')